In [1]:
#You might need to install these packages before running. Comment out after
!pip install torch
!pip install transformers datasets
!pip install scikit-learn


In [2]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/Rey/nlp_p"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/Rey/nlp_p


In [3]:
import csv
import pandas as pd


#Preprocessing the CSV file that contains the BASIL database.
df = pd.read_csv('processed_data.csv')

paragraphs = df["body"] # Gets paragraphs from CSV

#This line was intended to convert stances to integers. Is not needed anymore since now we use MultiLabel Binarizer for
# multilabel classification
# df['stance'] = df['stance'].replace([ 'left', 'center', 'liberal', 'conservative', 'right'],[0,1,2,3,4])


stances = df['stance'] # Gets all the stances/labels
# Stances: {'left', 'center', 'liberal', 'conservative', 'right'}
df['body'] =df['body'].astype(str)


# Use if you would want to print a sample paragraph and label
# print(df['body'][100])
# print(df['stance'][100])

In [4]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM,AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import TensorDataset, DataLoader


def multi_label_formatting(data):
    """This function does not really play a huge role for us right now, but what it basically does is that if a paragraph
    has two stances, i.e. 'conservative' and 'right', it will keep both for the classification problem.   """
    labels = []
    for i in range(len(data['stance'])):
        multi_tags = []
        if type(data['stance'][i]) is not float:
            multi_tags.append(data['stance'][i].lower())

        labels.append(multi_tags)

    return labels


#This package will convert tags to an array of size 5 (five because we have 5 stances:
# 'left', 'center', 'liberal', 'conservative', 'right') where, for example, if a paragraph is classified as 'center'
# it converts its label into one hot encoding [0,0,1,0,0]
mlb = MultiLabelBinarizer()
labels = multi_label_formatting(df) # In case of multitags. Look at function description for more info
print(f"Labels : {labels}")
#One Hot Enconding of Multi labels
labels = mlb.fit_transform(labels)

#Splitting data into test set and training set.
x_train_og, x_test_og, y_train, y_test = train_test_split(df['body'].astype(str), labels,test_size=0.10, random_state = 0)

#These following two models are way bigger and perform worse (tested.)
# model_name = "roberta-large"
# model_name = "roberta-base"

model_name = "launch/POLITICS" # POLITICS model from HuggingFace!

tokenizer = AutoTokenizer.from_pretrained(model_name)

# You can check that maximum amount of tokes is 512 which means that we will not be able
# to process the entire paragraphs.
# print(tokenizer.model_max_length)

# model = AutoModelForMaskedLM.from_pretrained("launch/POLITICS")
model = AutoModelForSequenceClassification.from_pretrained (model_name,num_labels=5) # num_labels = 5 enables hugging face to add a classification head to the model



train_encodings = tokenizer(x_train_og.to_list(), truncation=True, padding=True, return_tensors="pt")
train_labels = torch.tensor(y_train, dtype=torch.float32)
train_dataset = TensorDataset(train_encodings.input_ids, train_encodings.attention_mask, train_labels)
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)


#Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)


# # Fine-tuning loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()



    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss}")
    # test_model() Use if you would want to take a look at how the model is currently doing on the test data. BAD PRACTICE!
    if total_loss < 2:
      break


Labels : [['center'], ['right'], ['left'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['right'], ['center'], ['center'], ['conservative'], ['liberal'], ['center'], ['center'], ['center'], ['liberal'], ['right'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['conservative'], ['right'], ['liberal'], ['left'], ['right'], ['liberal'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['center'], ['right'], ['center'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['center'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['center'], ['conservative'], ['conservative'], ['right'], ['center'], ['left'], ['right'], ['center'], ['liberal'], ['right'], ['right'], ['liberal'], ['center'], ['right'], ['left'], ['right'], ['liberal'], ['conservative'], ['liberal'], ['center'], ['center'

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.bias', 'classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/50, Loss: 13.181722551584244
Epoch 2/50, Loss: 12.04808059334755
Epoch 3/50, Loss: 11.947849869728088
Epoch 4/50, Loss: 10.878373235464096
Epoch 5/50, Loss: 9.135710969567299
Epoch 6/50, Loss: 7.524197727441788
Epoch 7/50, Loss: 5.4522663950920105
Epoch 8/50, Loss: 3.8166702315211296
Epoch 9/50, Loss: 2.1546063870191574
Epoch 10/50, Loss: 1.512246886268258


In [6]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np


# Dataloader for test data
test_encodings = tokenizer(x_test_og.to_list(), truncation=True, padding=True, return_tensors="pt")
test_labels = torch.tensor(y_test, dtype=torch.float32)
test_dataset = TensorDataset(test_encodings.input_ids, test_encodings.attention_mask, test_labels)
test_loader = DataLoader(test_dataset, batch_size=30, shuffle=False)  # No need to shuffle test data



# Predict on the test data
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits



        # Softmax makes more sense for single classifications
        predictions = outputs.logits.softmax(dim=-1).tolist()
        all_preds.extend(predictions)

        # In case you'd want to use Sigmoid
        # predictions = torch.sigmoid(logits)  # Apply sigmoid activation for multilabel classification
        # all_preds.extend(predictions.cpu().detach().numpy())

        all_labels.extend(labels.cpu().detach().numpy())

# Convert the predictions and labels to binary values based on a threshold (e.g., 0.5)
threshold = 0.5


all_preds = (torch.tensor(all_preds) > threshold).int().numpy()
all_labels = np.array(all_labels)

# Compute the classification report
report = classification_report(all_labels, all_preds, target_names=mlb.classes_)
accuracy = accuracy_score(all_labels, all_preds)

# Reporting Results
print(f"Model Accuracy : {accuracy}")

#Bigger report summary. Sample avg is the same as Accuracy.
print(report)


Model Accuracy : 0.5333333333333333
              precision    recall  f1-score   support

           0       0.55      0.92      0.69        13
           1       0.50      0.50      0.50         2
           2       0.00      0.00      0.00         3
           3       0.50      0.14      0.22         7
           4       0.67      0.40      0.50         5

   micro avg       0.55      0.53      0.54        30
   macro avg       0.44      0.39      0.38        30
weighted avg       0.50      0.53      0.47        30
 samples avg       0.53      0.53      0.53        30



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [5]:
#Memory Management
del df
del test_labels
del model
del tokenizer
torch.cuda.empty_cache()